In [181]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [182]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

1.

In [183]:
data_gdynia["number_of_storeys"].agg(["min", "max", "mean", "median"])

min        1.000000
max       38.000000
mean       3.790224
median     4.000000
Name: number_of_storeys, dtype: float64

In [184]:
data_gdynia["number_of_storeys"].quantile([0.25, 0.75, 0.9])

0.25    1.0
0.75    6.0
0.90    7.0
Name: number_of_storeys, dtype: Float64

In [185]:
data_gdynia["number_of_storeys"].value_counts()[:1]

number_of_storeys
1    379
Name: count, dtype: Int64

In [186]:
(data_gdynia["number_of_storeys"] >= 5).mean()

np.float64(0.47963340122199594)

In [187]:
data_gdynia.loc[data_gdynia["number_of_storeys"].idxmax()]

osm_id                                                 76208008,226567211
addr_street                                      Antoniego Hryniewickiego
addr_number                                                             6
ground_floor_group                                                    NaN
ground_floor_function    dentysta,dermatolog,restauracja,centrum medyczne
upper_floor_group                                                     NaN
upper_floor_function                           mieszkaniowa wielorodzinna
number_of_storeys                                                      38
is_mixed_use                                                         <NA>
build_year_est                                                       2000
arch_style                                                  współczesność
condition                                                               3
max_height                                                            NaN
is_heritage                           

In [188]:
assert np.isclose(
    data_gdynia["number_of_storeys"].agg(["min", "max", "mean", "median"]),
    [1, 38, 3.79, 4],
    atol=0.01,
).all()
assert (data_gdynia["number_of_storeys"].quantile([0.25, 0.75, 0.9]) == [1, 6, 7]).all()

assert (
    data_gdynia["number_of_storeys"]
    .value_counts()
    .sort_values(ascending=False)[:1]
    .index
    == 1
)
assert (
    data_gdynia["number_of_storeys"].value_counts().sort_values(ascending=False)[:1]
    == 379
).all()

assert np.isclose((data_gdynia["number_of_storeys"] >= 5).mean(), 0.4796, atol=0.01)

assert data_gdynia["number_of_storeys"].idxmax() == 257
assert (
    data_gdynia.loc[
        data_gdynia["number_of_storeys"].idxmax(), ["addr_street", "addr_number"]
    ]
    == ["Antoniego Hryniewickiego", "6"]
).all()

2.

In [189]:
memory_diff = (
    pd.concat(
        {
            "before": data_gdynia.memory_usage(deep=True),
            "after": data_gdynia.astype(
                {
                    "addr_street": "category",
                    "source": "category",
                    "arch_style": "category",
                    "upper_floor_function": "category",
                    "ground_floor_function": "category",
                    "osm_id": "category",
                }
            ).memory_usage(deep=True),
        },
        axis=1,
    )
    .drop("Index")
    .assign(ratio=lambda d: d["before"] / d["after"])
    .assign(nunique=lambda d: data_gdynia.nunique())
    .sort_values("ratio", ascending=False)
)

memory_diff

,before,after,ratio,nunique
source,62848,1046,60.084130,1
arch_style,72511,1333,54.396849,5
addr_street,64543,4574,14.110844,44
upper_floor_function,55389,6569,8.431877,78
ground_floor_function,77266,42018,1.838879,382
build_year_est,8838,8838,1.000000,13
upper_floor_group,31424,31424,1.000000,0
number_of_storeys,8838,8838,1.000000,15
ground_floor_group,31424,31424,1.000000,0
addr_number,44156,44156,1.000000,179


When all elements are unique saving as categories doesnt have sense, becouse it takes more memory. The smaller nunique the less memory the series takes.

3.

In [190]:
categories = {1: "low", 2: "medium", 3: "high"}
data_conf_unordered = (
    data_gdynia["confidence"].map(categories).value_counts().sort_index()
)
data_conf_unordered

confidence
high      317
low        19
medium    646
Name: count, dtype: int64

In [191]:
conf_type = pd.CategoricalDtype(categories=["low", "medium", "high"], ordered=True)
data_conf_oredered = data_gdynia["confidence"].map(categories).astype(conf_type)
data_conf_oredered.value_counts().sort_index()

confidence
low        19
medium    646
high      317
Name: count, dtype: int64

In [192]:
assert ((data_conf_oredered.value_counts().sort_index()) == [19, 646, 317]).all()
assert (data_conf_oredered >= "medium").sum() == 963
assert data_conf_oredered.max() == "high"

4.

In [ ]:
data_gdynia.assign(
    confidence_level=lambda d: d["confidence"].map(categories).astype(conf_type)
).assign(is_high=lambda d: d["confidence_level"] == "high").groupby("arch_style").agg(
    num_buildings=("osm_id", "size"), share_high=("is_high", "mean")
)


,num_buildings,share_high
arch_style,,
klasycyzm,49,0.142857
modernizm,466,0.143777
postmodernizm,4,0.250000
tradycjonalizm,7,0.285714
współczesność,456,0.526316


5.

In [ ]:
arch_style_addr_street_unstack_obs = (
    data_gdynia.astype({"arch_style": "category"})
    .loc[data_gdynia["addr_street"].isin(["Świętojańska", "Portowa", "Armii Krajowej"])]
    .groupby(["addr_street", "arch_style"], observed=False)
    .size()
    .unstack()
)
arch_style_addr_street_unstack_obs

arch_style,klasycyzm,modernizm,postmodernizm,tradycjonalizm,współczesność
addr_street,,,,,
Armii Krajowej,1,19,0,0,5
Portowa,0,10,0,0,3
Świętojańska,9,90,0,0,7


In [ ]:
arch_style_addr_street_unstack = (
    data_gdynia.astype({"arch_style": "category"})
    .loc[lambda d: d["addr_street"].isin(["Świętojańska", "Portowa", "Armii Krajowej"])]
    .groupby(["addr_street", "arch_style"])
    .size()
    .unstack()
)
arch_style_addr_street_unstack

arch_style,klasycyzm,modernizm,współczesność
addr_street,,,
Armii Krajowej,1.0,19.0,5.0
Portowa,NaN,10.0,3.0
Świętojańska,9.0,90.0,7.0


In [217]:
arch_style_addr_street_unstack.sum(axis=1)

addr_street
Armii Krajowej     25.0
Portowa            13.0
Świętojańska      106.0
dtype: float64

In [ ]:
assert (arch_style_addr_street_unstack_obs == 0).sum().sum() == 7
assert arch_style_addr_street_unstack.sum(axis=1).to_dict() == {
    "Armii Krajowej": 25,
    "Portowa": 13,
    "Świętojańska": 106,
}